# RaptGen
In this notebook, we present a tutorial on using RaptGen [[1](#ref-1)] for the following tasks:
1. **Embed aptamers in a latent space:** given a pool of sequences, map each one to a point in a low-dimensional space, where similar sequences lie close together;
2. **Generate aptamer motifs:** decode a point of that space into a sequence motif.

RaptGen is a variational autoencoder with two parts:

- The **encoder** maps a sequence to a point in the latent space.
- The **decoder** turns a point back into a profile HMM. A profile HMM is a model of a motif. It gives the base probabilities at each position, and it allows insertions and deletions. A sampler converts it to sequences.

The classes used in this notebook are `MoleculeLoader`, a lazy container holding sequences or file paths; `SequenceIndexEncoder`, which encodes a sequence as one integer per base; `RaptGenModel`, the estimator that trains the autoencoder and decodes latent points; and `RaptGenGenerator`, the end-to-end algorithm that ties them together.

The tutorial is organized as follows. In [Step 1](#step-1-data-preparation) we prepare a small set of sequences. In [Step 2](#step-2-training-and-generation) we train the model and decode motifs from it. [Step 3](#step-3-settings-and-decoding) covers the settings that shape the motif and the two decoding modes, and [Step 4](#step-4-other-input-formats) the accepted input formats. Finally, [Step 5](#step-5-real-selex-data) repeats the workflow on a real SELEX dataset [[2](#ref-2)].

##### References
<a id="ref-1"></a>
[1] Iwano, Natsuki, et al. "Generative aptamer discovery using RaptGen." Nature Computational Science 2.6 (2022): 378-386. <br>
<a id="ref-2"></a>
[2] Jolma, Arttu, et al. "DNA-binding specificities of human transcription factors." Cell 152.1-2 (2013): 327-339. <br>

## Step 1: Data preparation
Here, we build a small set of aptamer sequences to train on.

Every sequence must have the same length and use the letters `A`, `C`, `G`, `T` or `U`. There is no `y`, because RaptGen trains on the sequences alone.

We use 5 RNA aptamers of 38 bases, repeated 10 times to give the trainer enough rows.

In [ ]:
import pandas as pd

from pyaptamer.data import MoleculeLoader
from pyaptamer.raptgen import RaptGenGenerator, RaptGenModel

In [ ]:
aptamer_sequence = [
    "GGGAGGACGAAGACGACUCGAGACAGGCUAGGGAGGGA",
    "AAGCGUCGGAUCUACACGUGCGAUAGCUCAGUACGCGG",
    "CGGUAUCGAGUACAGGAGUCCGACGGAUAGUCCGGAGC",
    "UAGCUAGCGAACUAGGCGUAGCUUCGAGUAGCUACGGA",
    "GCUAGGACGAUCGCACGUGACCGUCAGUAGCGUAGGAG",
]

sequences = aptamer_sequence * 10
X = MoleculeLoader(data={"aptamer": sequences})

In [ ]:
X.to_dataframe().head()

## Step 2: Training and generation
Here, we train the model and decode motifs from the latent space.

`RaptGenGenerator` runs the whole workflow. It encodes the sequences with `SequenceIndexEncoder` and trains a `RaptGenModel` on them. Pass your own `RaptGenModel` with `estimator=` to control training, or leave it out for the defaults, which run up to 1000 epochs.

The settings below are smaller than the defaults, so the demo trains in a few seconds. Leave them out on real data.

In [ ]:
model = RaptGenModel(
    motif_len=5,
    max_epochs=100,
    patience=20,
    force_epochs=20,
    random_state=0,
)
generator = RaptGenGenerator(estimator=model)

In [ ]:
generator.fit(X)

In [ ]:
Z = generator.transform(X)
print(Z.shape)

In [ ]:
generator.inverse_transform(Z[:3], most_likely=True)

With `most_likely=True`, the decoder follows the most probable path through the profile HMM, tries every A/C/G/T filling of the undetermined (insert) positions, scores each candidate under the HMM and returns the best one. The HMM models a motif of length `motif_len`, so the generated sequences are short motifs of about that length, and not full-length aptamers.

## Step 3: Settings and decoding
`motif_len` sets the length of the motif the model learns, and `embed_size` the number of dimensions of the latent space.

In [ ]:
model = RaptGenModel(
    motif_len=6,
    embed_size=4,
    max_epochs=100,
    patience=20,
    force_epochs=20,
    random_state=0,
)
generator = RaptGenGenerator(estimator=model)
generator.fit(X)

Z = generator.transform(X)
print(Z.shape)

In [ ]:
generator.inverse_transform(Z[:3], most_likely=False)

## Step 4: Other input formats
Here, we pass a DataFrame instead of a `MoleculeLoader`.

If the sequence column is not called `aptamer`, give its name with `aptamer_col`.

In [ ]:
df = pd.DataFrame({"my_sequences": sequences})
generator = RaptGenGenerator(
    aptamer_col="my_sequences",
    estimator=RaptGenModel(
        motif_len=5, max_epochs=100, patience=20, force_epochs=20, random_state=0
    ),
)

generator.fit(df)
generator.inverse_transform(generator.transform(df)[:3])

## Step 5: Real SELEX data
Here, we repeat the workflow on the reads of a real SELEX experiment.

The data is round 4 of an HT-SELEX selection against the human transcription factor BARX1 [[2](#ref-2)], with a 20 nucleotide random region. Each round keeps the sequences that bind the target, so a late round is dominated by the motif while an early round is still near-random.

### Loading the reads
`MoleculeLoader` parses the FASTQ file and, with `tiling="samples"`, gives one row per read.

In [ ]:
loader = MoleculeLoader(
    data={"aptamer": ["data_selex/ERR188681.fastq"]}, tiling="samples"
)
reads = loader.to_dataframe()

print(f"{len(reads)} reads in the file")
reads.head()

In [ ]:
reads["aptamer"].str.len().value_counts().head()

### Cleaning the reads
Every read should be 20 bases of `A`, `C`, `G` or `T`. A sequencer writes `N` where it could not call a base, so we drop any read that holds another letter, and any read that is not 20 long.

In [ ]:
seqs = reads["aptamer"].str.upper()
seqs = seqs[seqs.str.fullmatch("[ACGT]+")]
seqs = seqs[seqs.str.len() == 20]

print(f"{len(seqs)} reads left of {len(reads)}")

In [ ]:
X_real = seqs.sample(n=1000, random_state=0).to_frame()
X_real.head()

### Training and generating motifs
`fast=True` selects the faster of the two decoders. It defines the same model as the default one and trains several times quicker, so it suits a larger dataset like this.

`motif_len=8` asks for a motif of about 8 bases, which fits the `TAATTA` site plus some flanking positions. We then decode the first five latent points.

In [ ]:
model = RaptGenModel(
    motif_len=8,
    max_epochs=200,
    patience=200,
    force_epochs=20,
    random_state=0,
    fast=True,
)
generator_real = RaptGenGenerator(estimator=model)
generator_real.fit(X_real)

Z_real = generator_real.transform(X_real)
generator_real.inverse_transform(Z_real[:5], most_likely=True)

Each latent point decodes to a different variant of the same `TAAT` motif.

Two things to try on your own data. Decoding more latent points shows more of the variation the model found. Training on more reads, or for more epochs, spreads the latent space out, and a wider latent space gives more varied motifs.

### Visualising the latent space
The latent space has `embed_size` dimensions, so we project the embeddings of the reads to 2D with PCA. Reads with similar sequences lie close to each other.

The plot needs matplotlib, installed with `pip install "pyaptamer[plot]"`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Z2 = PCA(n_components=2).fit_transform(Z_real)

plt.figure(figsize=(5, 4))
plt.scatter(Z2[:, 0], Z2[:, 1], s=12, alpha=0.7)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("RaptGen latent space (PCA of the embeddings)")
plt.show()